In [1]:
import pandas as pd

# ── 경로 설정 ──────────────────────────────────────────────
INPUT_PATH  = r"D:\Project\★아파트지도\data\공동주택현황_최종.csv"
OUTPUT_PATH = r"D:\Project\★아파트지도\data\아파트_좌표정보.csv"

# ── 데이터 로드 ────────────────────────────────────────────
df = pd.read_csv(INPUT_PATH, encoding='utf-8-sig')
print(f"원본 행 수: {len(df):,}")

# ── 필요한 컬럼만 추출 ─────────────────────────────────────
cols = ['아파트명', '아파트코드', '세대수', '도로명주소', '위도', '경도']
df = df[cols].copy()

# ── 세대수 숫자 변환 (혹시 NaN 있으면 0으로) ──────────────
df['세대수'] = pd.to_numeric(df['세대수'], errors='coerce').fillna(0).astype(int)

# ── 그룹화 처리 ────────────────────────────────────────────
# 아파트명 + 아파트코드가 같으면:
#   - 세대수는 합산
#   - 도로명주소, 위도, 경도는 첫 번째 값 사용
agg_dict = {
    '세대수'    : 'sum',
    '도로명주소': 'first',
    '위도'      : 'first',
    '경도'      : 'first',
}
df_grouped = (
    df.groupby(['아파트명', '아파트코드'], as_index=False)
      .agg(agg_dict)
)

# ── 컬럼 순서 정리 ─────────────────────────────────────────
df_grouped = df_grouped[cols]

print(f"처리 후 행 수: {len(df_grouped):,}  (중복 제거 + 세대수 합산 완료)")

# ── 저장 ───────────────────────────────────────────────────
df_grouped.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f"저장 완료 → {OUTPUT_PATH}")

# ── 미리보기 ───────────────────────────────────────────────
print("\n[미리보기 상위 5행]")
print(df_grouped.head().to_string(index=False))

원본 행 수: 3,259
처리 후 행 수: 2,785  (중복 제거 + 세대수 합산 완료)
저장 완료 → D:\Project\★아파트지도\data\아파트_좌표정보.csv

[미리보기 상위 5행]
     아파트명  아파트코드  세대수           도로명주소        위도         경도
1차두성타운아파트  22605  285  북구 태전로7길 10-17 35.923942 128.542986
2군사령부간부숙소 201478   43      수성구 무열로 56 35.863234 128.649965
2군사령부참모관사 201218   10      수성구 무열로 56 35.863234 128.649965
 2차금류강남타운 200004  499 고령군 다산면 상곡3길 28 35.825473 128.447942
   2차대성빌라 200301   16     동구 해동로 97-1 35.890596 128.641596
